# OmniSub 2026 — Lip Reading + LRS2 Matching

In [ ]:
# ── Cell 1: Install & clone auto_avsr ──
import os, subprocess, glob

!pip install -q jiwer sentencepiece scikit-image av
!pip install -q mediapipe==0.10.14
!pip install -q 'Pillow>=10.0'

import mediapipe as mp
print(f'mediapipe {mp.__version__}, solutions={hasattr(mp, "solutions")}')

AVSR_DIR = '/kaggle/working/auto_avsr'
if not os.path.exists(AVSR_DIR):
    !git clone --depth 1 https://github.com/mpc001/auto_avsr.git {AVSR_DIR}

MODEL_PATH = None
for c in glob.glob('/kaggle/input/**/vsr_model.pth', recursive=True):
    if os.path.getsize(c) > 1e6:
        MODEL_PATH = c
        break
if not MODEL_PATH:
    for c in glob.glob('/kaggle/input/**/*.pth', recursive=True):
        if os.path.getsize(c) > 1e8:
            MODEL_PATH = c
            break

print(f'Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)' if MODEL_PATH else 'Model NOT FOUND')
print('Setup OK')

In [ ]:
# ── Cell 2: Discover paths & load LRS2 data ──
import os, csv, re, json, subprocess, time, sys, argparse
from pathlib import Path
from collections import defaultdict

def find_file(root, name, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= max_depth:
            dirnames.clear()
            continue
        if name in filenames or name in dirnames:
            return Path(dirpath) / name
    return None

SAMPLE_SUB = find_file('/kaggle/input', 'sample_submission.csv')
LRS2_DIR = find_file('/kaggle/input', 'lrs2_train_text.txt').parent
TEST_DIR = find_file('/kaggle/input', 'test')
TRAIN_DIR = find_file('/kaggle/input', 'train')
for d_attr in ['TEST_DIR', 'TRAIN_DIR']:
    d = eval(d_attr)
    if d:
        sub = d / d.name
        if sub.exists(): exec(f'{d_attr} = sub')
OUTPUT = Path('/kaggle/working/submission.csv')
print(f'TEST: {TEST_DIR}\nTRAIN: {TRAIN_DIR}\nLRS2: {LRS2_DIR}')

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

lrs2_by_channel = defaultdict(list)
for fname in ['lrs2_train_text.txt', 'lrs2_test_text.txt', 'lrs2_val_text.txt']:
    fpath = LRS2_DIR / fname
    if not fpath.exists(): continue
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split(' ', 1)
            if len(parts) < 2: continue
            ch = parts[0].rsplit('_', 1)[0]
            lrs2_by_channel[ch].append(norm(parts[1]))
for ch in lrs2_by_channel:
    lrs2_by_channel[ch] = list(dict.fromkeys(lrs2_by_channel[ch]))
lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
print(f'LRS2: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels')

# Load train txt files into candidate pool
if TRAIN_DIR and TRAIN_DIR.exists():
    train_added = 0
    for ch_name in os.listdir(TRAIN_DIR):
        ch_dir = TRAIN_DIR / ch_name
        if not ch_dir.is_dir(): continue
        existing = set(lrs2_by_channel.get(ch_name, []))
        for txt_file in ch_dir.glob('*.txt'):
            with open(txt_file) as f:
                first_line = f.readline().strip()
            if first_line.startswith('Text:'):
                text = norm(first_line[5:].strip())
                if text and text not in existing:
                    lrs2_by_channel[ch_name].append(text)
                    existing.add(text)
                    train_added += 1
    lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
    print(f'Train data: added {train_added} texts, now {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels')

test_paths = []
with open(SAMPLE_SUB) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader: test_paths.append(row[0])
paths_with_cand = [p for p in test_paths if p.split('/')[1] in lrs2_by_channel]
paths_no_cand = [p for p in test_paths if p.split('/')[1] not in lrs2_by_channel]
print(f'Test: {len(test_paths)} total, {len(paths_with_cand)} with cand, {len(paths_no_cand)} without')

In [ ]:
# ── Cell 3: Initialize auto_avsr lip reading pipeline (returns features + hypotheses) ──
import torch, torchvision, numpy as np
import torch.nn.functional as F

AVSR_DIR = '/kaggle/working/auto_avsr'
sys.path.insert(0, AVSR_DIR)

VSR_OK = False
pipeline = None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    try:
        from lightning import ModelModule, get_beam_search_decoder
        from datamodule.transforms import VideoTransform, TextTransform
        from preparation.detectors.mediapipe.detector import LandmarksDetector
        from preparation.detectors.mediapipe.video_process import VideoProcess
        from espnet.nets.pytorch_backend.transformer.mask import subsequent_mask

        class VSRPipeline(torch.nn.Module):
            def __init__(self, model_path, device='cuda'):
                super().__init__()
                self.device = device
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
                self.video_transform = VideoTransform(subset='test')
                args = argparse.Namespace()
                args.modality = 'video'
                args.ctc_weight = 0.1
                ckpt = torch.load(model_path, map_location='cpu')
                self.modelmodule = ModelModule(args)
                self.modelmodule.model.load_state_dict(ckpt)
                self.modelmodule.eval()
                if device == 'cuda' and torch.cuda.is_available():
                    self.modelmodule = self.modelmodule.cuda()
                self.beam_search = get_beam_search_decoder(
                    self.modelmodule.model, self.modelmodule.token_list
                )
                self.text_transform = self.modelmodule.text_transform
                self.model = self.modelmodule.model

            @torch.no_grad()
            def __call__(self, video_path):
                video_path = os.path.abspath(video_path)
                video = torchvision.io.read_video(video_path, pts_unit='sec')[0].numpy()
                landmarks = self.landmarks_detector(video)
                video = self.video_process(video, landmarks)
                if video is None:
                    return {'hypotheses': [''], 'ctc_logprobs': None, 'enc_feat': None}
                video = torch.tensor(video).permute(0, 3, 1, 2)
                video = self.video_transform(video)
                if self.device == 'cuda' and torch.cuda.is_available():
                    video = video.cuda()

                # Run encoder
                x = self.model.frontend(video.unsqueeze(0))
                x = self.model.proj_encoder(x)
                enc_feat, _ = self.model.encoder(x, None)
                enc_feat = enc_feat.squeeze(0)  # (T, D)

                # CTC log-probs for candidate scoring
                ctc_logprobs = self.model.ctc.log_softmax(enc_feat.unsqueeze(0)).squeeze(0)  # (T, V)

                # CTC greedy decode (bonus hypothesis)
                ctc_argmax = torch.argmax(ctc_logprobs, dim=-1)  # (T,)
                tokens = []
                prev = 0
                for t in ctc_argmax:
                    t_val = t.item()
                    if t_val != 0 and t_val != prev:
                        tokens.append(t_val)
                    prev = t_val
                ctc_text = ''
                if tokens:
                    ctc_text = self.text_transform.post_process(torch.tensor(tokens)).replace("<eos>", "")

                # Beam search → top 40 unique hypotheses
                nbest_hyps = self.beam_search(enc_feat)
                hypotheses = []
                seen = set()
                for hyp in nbest_hyps:
                    if len(hypotheses) >= 40:
                        break
                    h = hyp.asdict()
                    token_ids = torch.tensor(list(map(int, h["yseq"][1:])))
                    text = self.text_transform.post_process(token_ids).replace("<eos>", "")
                    if text.strip() and text not in seen:
                        hypotheses.append(text)
                        seen.add(text)

                # Add CTC greedy if not already present
                if ctc_text.strip() and ctc_text not in seen:
                    hypotheses.append(ctc_text)

                if not hypotheses:
                    hypotheses = ['']

                return {
                    'hypotheses': hypotheses,
                    'ctc_logprobs': ctc_logprobs,  # (T, V)
                    'enc_feat': enc_feat,           # (T, D)
                }

        print('Loading VSR model...')
        pipeline = VSRPipeline(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
        print('VSR pipeline ready')

        parts = test_paths[0].split('/')
        mp4_test = str(TEST_DIR / parts[1] / parts[2])
        print(f'Test: {mp4_test}')
        result = pipeline(mp4_test)
        print(f'Top-1: "{result["hypotheses"][0]}"')
        print(f'N-hyps: {len(result["hypotheses"])}, ctc_logprobs: {result["ctc_logprobs"].shape if result["ctc_logprobs"] is not None else None}, enc_feat: {result["enc_feat"].shape if result["enc_feat"] is not None else None}')
        VSR_OK = True

    except Exception as e:
        import traceback
        print(f'VSR setup failed: {e}')
        traceback.print_exc()
        VSR_OK = False
else:
    print('No model — skipping VSR')

print(f'\nVSR_OK: {VSR_OK}')

In [ ]:
# ── Cell 4: Transcribe all test videos + two-pass candidate scoring ──
vsr_results = {}

TOP_K = 15  # Candidates to pass from CTC to attention scoring

def compute_ctc_score(ctc_logprobs, token_ids, device):
    """CTC forward probability: P(candidate | visual). Length-normalized. Higher = better."""
    T = ctc_logprobs.size(0)
    S = token_ids.size(0)
    if S == 0 or S > T:
        return float('-inf')
    log_probs = ctc_logprobs.unsqueeze(1)  # (T, 1, V)
    loss = F.ctc_loss(
        log_probs, token_ids.unsqueeze(0),  # (1, S)
        torch.tensor([T], device=device), torch.tensor([S], device=device),
        blank=0, reduction='none', zero_infinity=True
    )
    return -loss.item() / max(S, 1)  # length-normalized log-prob

def compute_att_scores_batch(model, enc_feat, candidates_token_ids, device):
    """Score multiple candidates in a single decoder forward pass."""
    if not candidates_token_ids:
        return []
    eos = model.odim - 1
    K = len(candidates_token_ids)

    # Prepare: SOS + tokens, padded to max length with eos
    max_len = max(len(t) for t in candidates_token_ids) + 1  # +1 for SOS
    tgt_in = torch.full((K, max_len), eos, device=device, dtype=torch.long)
    lengths = []
    targets_all = []
    for i, tids in enumerate(candidates_token_ids):
        seq = [eos] + tids
        tgt_in[i, :len(seq)] = torch.tensor(seq, device=device)
        lengths.append(len(tids) + 1)
        targets_all.append(tids + [eos])

    # Causal mask only (padding positions don't affect earlier positions)
    tgt_mask = subsequent_mask(max_len, device=device).unsqueeze(0)
    memory = enc_feat.unsqueeze(0).expand(K, -1, -1)

    logits, _ = model.decoder.forward(tgt_in, tgt_mask, memory, None)
    log_probs = F.log_softmax(logits, dim=-1)

    scores = []
    for i in range(K):
        targets = targets_all[i]
        n = len(targets)
        score = sum(log_probs[i, j, targets[j]].item() for j in range(n)) / n
        scores.append(score)
    return scores

if VSR_OK:
    model = pipeline.model
    text_transform = pipeline.text_transform
    device = next(model.parameters()).device

    print(f'Transcribing {len(test_paths)} videos + two-pass scoring (TOP_K={TOP_K})...')
    start = time.time()
    errors = 0
    scored_count = 0

    for i, path in enumerate(test_paths):
        parts = path.split('/')
        mp4_path = str(TEST_DIR / parts[1] / parts[2])
        ch = parts[1]
        candidates = lrs2_by_channel.get(ch, [])

        try:
            result = pipeline(mp4_path)
            hypotheses = [norm(str(h)) for h in result['hypotheses'] if h]
            if not hypotheses:
                hypotheses = ['']
            ctc_logprobs = result['ctc_logprobs']
            enc_feat = result['enc_feat']
        except:
            hypotheses = ['']
            ctc_logprobs = None
            enc_feat = None
            errors += 1

        clip_data = {'hypotheses': hypotheses}

        if ctc_logprobs is not None and enc_feat is not None and candidates:
            # ── Pass 1: CTC-score ALL candidates (fast, ~1ms each) ──
            ctc_scores = {}
            token_ids_cache = {}
            for cand in candidates:
                token_ids = text_transform.tokenize(cand)
                if device.type == 'cuda':
                    token_ids = token_ids.cuda()
                token_ids_cache[cand] = token_ids
                ctc_scores[cand] = compute_ctc_score(ctc_logprobs, token_ids, device)

            # Select top-K for attention scoring
            top_k_cands = sorted(ctc_scores, key=ctc_scores.get, reverse=True)[:TOP_K]

            # Inject beam search top-1 match if not already in top-K
            hyp_set = set(h for h in hypotheses[:5] if h.strip())
            for cand in candidates:
                if cand in hyp_set and cand not in top_k_cands:
                    top_k_cands.append(cand)
                    break

            # ── Pass 2: Attention-score top-K only (batched) ──
            top_k_token_ids = [token_ids_cache[c].tolist() for c in top_k_cands]
            att_score_list = compute_att_scores_batch(model, enc_feat, top_k_token_ids, device)
            att_scores = dict(zip(top_k_cands, att_score_list))

            clip_data['ctc_scores'] = ctc_scores
            clip_data['att_scores'] = att_scores
            scored_count += 1

        elif ctc_logprobs is not None and not candidates:
            # No-candidate clip: store ctc_logprobs on CPU for later global scoring
            clip_data['ctc_logprobs_stored'] = ctc_logprobs.cpu()

        # Free GPU tensors immediately
        del ctc_logprobs, enc_feat

        vsr_results[path] = clip_data

        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i+1) / elapsed
            eta = (len(test_paths) - i - 1) / rate / 60 if rate > 0 else 0
            ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
            print(f'  [{i+1}/{len(test_paths)}] {rate:.2f}/s ETA {eta:.0f}min ok={ok} err={errors} scored={scored_count} nhyps={len(hypotheses)} | top1="{hypotheses[0][:50]}"')
        if (i+1) % 500 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
    nhyps_avg = sum(len(v['hypotheses']) for v in vsr_results.values()) / max(len(vsr_results), 1)
    print(f'\nDone: {ok}/{len(vsr_results)} ok, {errors} err, scored={scored_count}, avg_hyps={nhyps_avg:.1f}, {(time.time()-start)/60:.1f}min')
else:
    print('VSR not available')

In [ ]:
# ── Cell 5: Combined ranking — direct log-prob combination (no min-max) ──
from jiwer import wer as compute_wer, cer as compute_cer

def match_score(ref, hyp):
    """Combined WER+CER score — CER weighted higher for short-sentence robustness."""
    try:
        w = compute_wer(ref, hyp)
        c = compute_cer(ref, hyp)
        return 0.4 * w + 0.6 * c
    except:
        return 1.0

TEXT_MATCH_HYPS = 10
W_CTC = 0.35
W_ATT = 0.55
W_TEXT = 0.10
EXACT_BONUS = 0.5

HAS_VSR = bool(vsr_results) and sum(1 for v in vsr_results.values() if v['hypotheses'][0]) > 100
print(f'Using VSR: {HAS_VSR}')
results = {}
stats = {'model_pick': 0, 'vsr_only': 0, 'duration': 0, 'global': 0, 'empty': 0, 'no_cand_ctc': 0}

if HAS_VSR:
    start = time.time()

    # ── Clips WITH channel candidates: direct log-prob combination ──
    for i, path in enumerate(paths_with_cand):
        ch = path.split('/')[1]
        candidates = lrs2_by_channel[ch]
        clip_data = vsr_results.get(path, {'hypotheses': ['']})
        hypotheses = clip_data['hypotheses']
        ctc_scores = clip_data.get('ctc_scores', {})
        att_scores = clip_data.get('att_scores', {})

        if not hypotheses[0]:
            results[path] = candidates[0]
            stats['duration'] += 1
            continue

        has_model_scores = bool(ctc_scores)

        if has_model_scores:
            text_hyps = [h for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            hyp_set = set(h for h in hypotheses[:10] if h.strip())

            # Score candidates that have attention scores (top-K)
            best_cand, best_score = candidates[0], float('-inf')
            for cand in att_scores:
                ctc_s = ctc_scores.get(cand, float('-inf'))
                att_s = att_scores[cand]
                text_s = min((match_score(cand, h) for h in text_hyps), default=1.0) if text_hyps else 1.0
                exact_bonus = EXACT_BONUS if cand in hyp_set else 0.0
                combined = W_CTC * ctc_s + W_ATT * att_s - W_TEXT * text_s + exact_bonus
                if combined > best_score:
                    best_score = combined
                    best_cand = cand

            # Also check remaining candidates (CTC-only, no attention score)
            for cand in candidates:
                if cand in att_scores:
                    continue
                ctc_s = ctc_scores.get(cand, float('-inf'))
                if ctc_s == float('-inf'):
                    continue
                text_s = min((match_score(cand, h) for h in text_hyps), default=1.0) if text_hyps else 1.0
                exact_bonus = EXACT_BONUS if cand in hyp_set else 0.0
                combined_ctc_only = 0.60 * ctc_s - 0.40 * text_s + exact_bonus
                if combined_ctc_only > best_score:
                    best_score = combined_ctc_only
                    best_cand = cand

            results[path] = best_cand
            stats['model_pick'] += 1
        else:
            # Fallback: text-only matching (no model scores available)
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            best_text, best_score = candidates[0], float('inf')
            for hyp in text_hyps:
                for cand in candidates:
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0:
                            break
                if best_score == 0.0:
                    break
            results[path] = best_text
            stats['model_pick'] += 1

        if (i+1) % 500 == 0 or i == 0:
            print(f'  [{i+1}/{len(paths_with_cand)}] | {stats}')

    # ── Clips WITHOUT channel candidates: CTC scoring against filtered global pool ──
    wc_index = defaultdict(list)
    for t in lrs2_all_texts:
        wc_index[len(t.split())].append(t)

    def trigrams(text):
        return set(text[i:i+3] for i in range(len(text)-2))

    for path in paths_no_cand:
        clip_data = vsr_results.get(path, {'hypotheses': ['']})
        hypotheses = clip_data['hypotheses']
        ctc_logprobs = clip_data.get('ctc_logprobs_stored')

        if not hypotheses[0]:
            results[path] = ''
            stats['empty'] += 1
            continue

        # Filter global pool by word count +-2
        w_wc = len(hypotheses[0].split())
        pool = []
        for wc in range(max(1, w_wc - 2), w_wc + 3):
            pool.extend(wc_index.get(wc, []))

        if not pool:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1
            continue

        # If we have CTC logprobs, use CTC scoring on filtered pool
        if ctc_logprobs is not None:
            # Character trigram pre-filter to reduce pool
            hyp_tri = trigrams(hypotheses[0])
            if hyp_tri:
                pool = [c for c in pool if len(trigrams(c) & hyp_tri) / max(len(hyp_tri), 1) > 0.15]
            pool = pool[:500]  # safety limit

            if pool:
                # Move logprobs to device for scoring
                lp = ctc_logprobs.to(device)
                best_cand, best_score = hypotheses[0], float('-inf')
                for cand in pool:
                    token_ids = text_transform.tokenize(cand)
                    if device.type == 'cuda':
                        token_ids = token_ids.cuda()
                    s = compute_ctc_score(lp, token_ids, device)
                    if s > best_score:
                        best_score = s
                        best_cand = cand
                del lp
                results[path] = best_cand
                stats['no_cand_ctc'] += 1
            else:
                results[path] = hypotheses[0]
                stats['vsr_only'] += 1
        else:
            # Fallback: text-only matching
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            best_text, best_score = pool[0], float('inf')
            for hyp in text_hyps:
                for cand in pool:
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0:
                            break
                if best_score == 0.0:
                    break

            if best_score < 0.25:
                results[path] = best_text
                stats['global'] += 1
            else:
                results[path] = hypotheses[0]
                stats['vsr_only'] += 1

    print(f'Done {(time.time()-start)/60:.1f}min | {stats}')
else:
    print('DURATION FALLBACK')
    def get_dur(mp4):
        try:
            r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(mp4)], capture_output=True, text=True, timeout=10)
            return float(r.stdout.strip())
        except: return None
    wps_s, cps_s = [], []
    if TRAIN_DIR and TRAIN_DIR.exists():
        count = 0
        for ch_name in sorted(os.listdir(TRAIN_DIR)):
            ch_dir = TRAIN_DIR / ch_name
            if not ch_dir.is_dir(): continue
            for txt_file in ch_dir.glob('*.txt'):
                with open(txt_file) as f: line = f.readline().strip()
                if not line.startswith('Text:'): continue
                text = norm(line[5:].strip())
                mp4 = txt_file.with_suffix('.mp4')
                if not mp4.exists(): continue
                dur = get_dur(str(mp4))
                if dur and dur > 0.3:
                    wps_s.append(len(text.split())/dur)
                    cps_s.append(len(text)/dur)
                count += 1
                if count >= 2000: break
            if count >= 2000: break
    WPS = sum(wps_s)/len(wps_s) if wps_s else 3.15
    CPS = sum(cps_s)/len(cps_s) if cps_s else 15.76
    for path in test_paths:
        ch = path.split('/')[1]
        cands = lrs2_by_channel.get(ch, [])
        if not cands: results[path] = ''; stats['empty'] += 1; continue
        if len(cands) == 1: results[path] = cands[0]; stats['model_pick'] += 1; continue
        parts = path.split('/')
        dur = get_dur(str(TEST_DIR / parts[1] / parts[2]))
        if dur and dur > 0.3:
            ew, ec = dur * WPS, dur * CPS
            results[path] = min(cands, key=lambda t: 0.6*abs(len(t.split())-ew)/max(ew,1) + 0.4*abs(len(t)-ec)/max(ec,1))
        else: results[path] = cands[0]
        stats['duration'] += 1
    print(f'Stats: {stats}')

In [ ]:
# ── Cell 6: Write submission ──
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'transcription'])
    for path in test_paths:
        text = results.get(path, '')
        text = norm(text) if text else ''
        writer.writerow([path, text])
import pandas as pd
sub = pd.read_csv(OUTPUT)
print(f'Shape: {sub.shape}, Empty: {(sub["transcription"].isna() | (sub["transcription"] == "")).sum()}')
sub['wc'] = sub['transcription'].apply(lambda x: len(str(x).split()))
print(f'Mean words: {sub["wc"].mean():.1f}')
print(sub.head(10))
print(f'Written to {OUTPUT}')